# GMTRouter - Training

This notebook demonstrates how to train the **GMTRouter** (Graph-based Multi-Turn Router).

## Overview

GMTRouter uses a Heterogeneous Graph Neural Network (HeteroGNN) to model complex relationships
in multi-turn conversations for personalized LLM routing.

**Key Features**:
- 5 Node types: User, Session, Query, LLM, Response
- 21 Edge types capturing various relationships
- Personalized routing based on user preferences
- Multi-turn conversation support

**Requirements**:
- PyTorch 2.0+
- PyTorch Geometric (optional but recommended)
- GMTRouter dataset

## 1. Environment Setup

In [1]:
# Install required packages (for Colab)
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter
!pip install -e .
!pip install torch torch-geometric


Cloning into 'LLMRouter'...
remote: Enumerating objects: 6017, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 6017 (delta 98), reused 87 (delta 81), pack-reused 5845 (from 2)
Receiving objects: 100% (6017/6017), 89.41 MiB | 53.86 MiB/s, done.
Resolving deltas: 100% (2946/2946), done.
Updating files: 100% (288/288), done.
/home/zhongjie/LLMRouter
Obtaining file:///home/zhongjie/LLMRouter
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llmrouter-lib (pyproject.toml) ... done
  Created wheel for llmrouter-lib: filename=llmrouter_lib-0.2.0-0.editable-py3-none-any.whl size=15709 sha256=f1a7e8ab14b77420868477c073f2bc3a6fa03a5b0211b6853ae72a3c7f013d8e
  Stored in directory: /tmp/pip-ephem-wheel-cache-ln6sfduv/wheels/82/4a/fd/59c4aec93c356c380d

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [3]:
import torch
from llmrouter.models.gmtrouter import GMTRouter, GMTRouterTrainer
from llmrouter.utils import setup_environment

setup_environment()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## 2. Configuration

GMTRouter uses the following configuration parameters:

| Parameter | Description | Default |
|-----------|-------------|--------|
| `num_gnn_layers` | Number of HGT layers | 2 |
| `hidden_dim` | Hidden dimension | 128 |
| `dropout` | Dropout rate | 0.1 |
| `epochs` | Training epochs | 350 |
| `lr` | Learning rate | 5e-4 |
| `objective` | Training objective | "auc" |

In [4]:
import yaml

CONFIG_PATH = "configs/model_config_train/gmtrouter.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print("Current Configuration:")
print("=" * 50)
print(yaml.dump(config, default_flow_style=False))

Current Configuration:
checkpoint:
  root: ./models
  save_every: 25
config:
  preprocess: true
data_path:
  data_root: ./data
  test_set: ./data/mt_bench/test_set.jsonl
  training_set: ./data/mt_bench/training_set.jsonl
  valid_set: ./data/mt_bench/valid_set.jsonl
dataset:
  name: mt_bench
  path: ./data
gmt_config:
  dropout: 0.1
  hidden_dim: 128
  num_gnn_layers: 2
  personalization: true
model_path:
  checkpoint_root: ./saved_models/gmtrouter
  load_model_path: ./saved_models/gmtrouter/gmtrouter.pt
  save_model_path: ./saved_models/gmtrouter/gmtrouter.pt
train:
  binary: true
  epochs: 350
  eval_every: 5
  id: llmrouter_gmt
  lr: 5e-4
  objective: auc
  prediction_count: 256
  seed: 136



## 3. Data Preparation

GMTRouter requires specific data format. 

First, download the dataset to `./data/mt_bench`.

Then, extract the dataset in place:

In [11]:
!pwd

/home/zhongjie/LLMRouter


In [15]:
rm -rf /home/zhongjie/LLMRouter/data/mt_bench/GMTRouter_Dataset 

In [16]:
!tar -xzvf ./data/mt_bench/GMTRouter_dataset.tar.gz -C ./data/mt_bench

!mv "./data/mt_bench/GMTRouter Dataset/data/mt_bench/"* "./data/mt_bench/"

!rm -rf "./data/mt_bench/GMTRouter Dataset"

GMTRouter Dataset/
GMTRouter Dataset/data/
GMTRouter Dataset/data/chatbot_arena/
GMTRouter Dataset/data/chatbot_arena/test_set.jsonl
GMTRouter Dataset/data/chatbot_arena/training_set.jsonl
GMTRouter Dataset/data/chatbot_arena/valid_set.jsonl
GMTRouter Dataset/data/mt_bench/
GMTRouter Dataset/data/mt_bench/test_set.jsonl
GMTRouter Dataset/data/mt_bench/training_set.jsonl
GMTRouter Dataset/data/mt_bench/valid_set.jsonl
GMTRouter Dataset/data/mmlu/
GMTRouter Dataset/data/mmlu/test_set.jsonl
GMTRouter Dataset/data/mmlu/training_set.jsonl
GMTRouter Dataset/data/mmlu/valid_set.jsonl
GMTRouter Dataset/data/gsm8k/
GMTRouter Dataset/data/gsm8k/test_set.jsonl
GMTRouter Dataset/data/gsm8k/training_set.jsonl
GMTRouter Dataset/data/gsm8k/valid_set.jsonl
GMTRouter Dataset/LICENSE
GMTRouter Dataset/README.md
mv: cannot stat './data/mt_bench/GMTRouter_dataset/*': No such file or directory
rmdir: failed to remove './data/mt_bench/GMTRouter_dataset': No such file or directory


In [19]:
# Check if data exists
data_path = config.get('data_path', {}).get('data_root', './data')
dataset_name = config.get('dataset', {}).get('name', 'mt_bench')

expected_path = os.path.join(data_path, dataset_name)

if os.path.exists(expected_path):
    print(f"Dataset found at: {expected_path}")
    files = os.listdir(expected_path)
    print(f"Files: {files}")
else:
    print(f"Dataset not found at: {expected_path}")
    print("\nPlease download the GMTRouter dataset:")
    print("1. Download from Google Drive")
    print("2. Extract to ./data/")
    print("3. Expected structure:")
    print("   ./data/mt_bench/training_set.jsonl")
    print("   ./data/mt_bench/valid_set.jsonl")
    print("   ./data/mt_bench/test_set.jsonl")

Dataset found at: ./data/mt_bench
Files: ['.ipynb_checkpoints', 'GMTRouter_dataset.tar.gz', 'mt_bench_to_jsonl.py', 'test_set.jsonl', 'training_set.jsonl', 'valid_set.jsonl']


## 4. Initialize Router

In [6]:
router = GMTRouter(yaml_path=CONFIG_PATH)

print("Router initialized successfully!")
print(f"Number of GNN layers: {config['gmt_config']['num_gnn_layers']}")
print(f"Hidden dimension: {config['gmt_config']['hidden_dim']}")
print(f"Personalization: {config['gmt_config']['personalization']}")

✅ MetaRouter initialized successfully (YAML + data loaded).
No pretrained model found at ./saved_models/gmtrouter/gmtrouter.pt
GMTRouter will need to be trained first.
Router initialized successfully!
Number of GNN layers: 2
Hidden dimension: 128
Personalization: True


## 5. Graph Structure

GMTRouter builds a heterogeneous graph with:
- **User nodes**: User embeddings and preferences
- **Session nodes**: Conversation session representations
- **Query nodes**: Query embeddings
- **LLM nodes**: LLM model embeddings
- **Response nodes**: Response quality scores

In [ ]:
print("GMTRouter Graph Structure:")
print("=" * 50)
print("\nNode Types:")
print("  1. User     - User preferences and history")
print("  2. Session  - Conversation sessions")
print("  3. Query    - User queries")
print("  4. LLM      - Language models")
print("  5. Response - Model responses")
print("\nEdge Types (21 total):")
print("  - own/owned_by (User-Session)")
print("  - contains/contained_by (Session-Query)")
print("  - answered_by/answered_to (Query-Response)")
print("  - generated_by/generated (LLM-Response)")
print("  - next/prev (Query-Query temporal)")
print("  - ... and more")

## 6. Training

In [8]:
trainer = GMTRouterTrainer(router=router, device=device)

print("Trainer initialized!")
print(f"Device: {device}")

Trainer initialized!
Device: cuda


In [20]:
print("Starting training...")
print("=" * 50)
print("Note: GMTRouter training uses pairwise learning on the graph.")
print("This may take significant time for large datasets.")
print("=" * 50)

trainer.train()

print("=" * 50)
print("Training completed!")

Starting training...
Note: GMTRouter training uses pairwise learning on the graph.
This may take significant time for large datasets.
GMTRouter Training
Loading training data from ./data/mt_bench/training_set.jsonl...
Loading GMTRouter data from ./data/mt_bench/training_set.jsonl...
  2 validation errors for GMTRouterInteraction
conversation.0.rating
  Value error, Rating must be between 0 and 5 [type=value_error, input_value=-1.3121, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
conversation.1.rating
  Value error, Rating must be between 0 and 5 [type=value_error, input_value=-1.7922, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
  2 validation errors for GMTRouterInteraction
conversation.0.rating
  Value error, Rating must be between 0 and 5 [type=value_error, input_value=-0.842, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
co

ValueError: Data validation failed

## 7. Model Verification

In [ ]:
# Check saved model
save_path = config['model_path'].get('save_model_path', './saved_models/gmtrouter/gmtrouter.pt')

if os.path.exists(save_path):
    print(f"Model saved at: {save_path}")
    checkpoint = torch.load(save_path, map_location='cpu')
    print(f"Checkpoint keys: {checkpoint.keys() if isinstance(checkpoint, dict) else 'state_dict'}")
else:
    print(f"Model not found at: {save_path}")

In [ ]:
# Test prediction
test_query = {
    "query": "What is machine learning?",
    "user_id": "test_user",
    "session_id": "test_session"
}
result = router.route_single(test_query)

print(f"Test query: {test_query['query']}")
print(f"User: {test_query['user_id']}")
print(f"Routed to: {result['model_name']}")

## Summary

In this notebook, we:

1. **Loaded Configuration**: Set up GMTRouter with YAML configuration
2. **Understood Graph Structure**: 5 node types, 21 edge types
3. **Trained Model**: Pairwise learning on heterogeneous graph
4. **Verified Model**: Tested personalized routing

**Key Takeaways**:
- GMTRouter captures complex multi-turn conversation patterns
- User personalization improves routing quality
- HeteroGNN learns from relational structure

**Next Steps**:
- Use the next part of notebook for inference
- Experiment with different datasets

# GMTRouter - Inference

This part of notebook demonstrates how to use a trained **GMTRouter** for inference.

## 1. Environment Setup (optional)

In [ ]:
import torch
from llmrouter.models.gmtrouter import GMTRouter
from llmrouter.utils import setup_environment
import yaml

setup_environment()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Load Trained Router

In [ ]:
CONFIG_PATH = "configs/model_config_train/gmtrouter.yaml"

router = GMTRouter(yaml_path=CONFIG_PATH)
print("Router loaded!")

## 3. Personalized Query Routing

In [ ]:
# Multi-turn conversation example
CONVERSATION = [
    {
        "query": "What is machine learning?",
        "user_id": "user_001",
        "session_id": "session_001",
        "turn": 1
    },
    {
        "query": "Can you give me a practical example?",
        "user_id": "user_001",
        "session_id": "session_001",
        "turn": 2
    },
    {
        "query": "How do I implement this in Python?",
        "user_id": "user_001",
        "session_id": "session_001",
        "turn": 3
    },
]

print("Multi-turn Routing Results:")
print("=" * 60)

for query in CONVERSATION:
    result = router.route_single(query)
    print(f"Turn {query['turn']}: {query['query'][:40]}...")
    print(f"   Routed to: {result['model_name']}")

## 4. Different User Preferences

In [ ]:
# Same query, different users
query_text = "Explain neural networks."

users = ["user_001", "user_002", "user_003"]

print(f"Query: {query_text}")
print("=" * 60)

for user in users:
    query = {
        "query": query_text,
        "user_id": user,
        "session_id": f"{user}_session"
    }
    result = router.route_single(query)
    print(f"{user}: Routed to {result['model_name']}")

## 5. File-Based Inference

Load queries from a file and save results.

In [ ]:
import json

# Load queries from a JSONL file
def load_queries_from_file(file_path):
    """Load queries from a JSONL file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    return queries

# Save results to a JSONL file
def save_results_to_file(results, output_path):
    """Save routing results to a JSONL file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Results saved to: {output_path}")

# Example: Load from default query file
QUERY_FILE = "data/example_data/query_data/default_query_test.jsonl"
OUTPUT_FILE = "outputs/gmtrouter_results.jsonl"

if os.path.exists(QUERY_FILE):
    # Load queries
    file_queries = load_queries_from_file(QUERY_FILE)
    print(f"Loaded {len(file_queries)} queries from: {QUERY_FILE}")
    
    # Route queries
    file_results = router.route_batch(batch=file_queries[:10])
    print(f"Routed {len(file_results)} queries")
    
    # Save results
    save_results_to_file(file_results, OUTPUT_FILE)
    
    # Show sample results
    print(f"\nSample results:")
    for i, result in enumerate(file_results[:3], 1):
        print(f"  {i}. {result.get('query', '')[:40]}... -> {result['model_name']}")
else:
    print(f"Query file not found: {QUERY_FILE}")
    print("Create a JSONL file with format: {\"query\": \"Your question\"}")

## Summary

GMTRouter provides:
- Personalized routing based on user history
- Multi-turn context awareness
- Graph-based relationship modeling